## 파이토치 기본

### 파이토치 공식 튜토리얼
- https://tutorials.pytorch.kr/

### 파이토치 패키지(API)
- `torch` : 메인 네임스페이스. 텐서(기본단위) 등 수학함수 포함. Numpy와 유사한 구조
- `torch.autograd` : 자동미분을 위한 함수들 포함. 컨텍스트 매니저, 기반클래스 포함
- `torch.nn` / `torch.nn.functional` : neurelnetwork. 신경망 구축위한 데이터 구조, 레이어등이 정의되어 있는 모듈. ReLU 등도 포함되어있다
- `torch.optim` : SGD(확률적 경사 하강법) 중심 파라미터 최적화 알고리즘 포함
- `torch.utils` : SGD 반복 연산 시 배치용 유틸리티 함수 포함
- torch.multiprocessing : 파이토치용 병렬프로세싱 환경을 안전하게 다루기 위한 모듈 

### 텐서
- 파이토치의 기본단위
    - 1차원 배열 - 벡터
    - 2차원 배열 - 행렬(매트릭스)
    - 3차원 배열 - 텐서

- <img src='../image/ml0018.png' width='600'>

- <img src='../image/ml0019.png' width='600'>

In [3]:
import torch
import numpy as np

In [2]:
t1 = torch.tensor([1.0, 2.0, 3.0])
t1

tensor([1., 2., 3.])

In [ ]:
# 텐서의 크기, 텐서자료형, 자료형+전체타입, 어느디바이스에서 사용중
print(t1.shape, t1.dtype, t1.type(), t1.device)

torch.Size([3]) torch.float32 torch.FloatTensor cpu


In [5]:
n1 = np.array([1.0, 2.0, 3.0])
n1

array([1., 2., 3.])

In [19]:
print(n1.shape, n1.dtype)

(3,) float64


In [6]:
## numpy에서 1, 0으로 초기화 되는 
n2 = np.ones([2,3])
n2

array([[1., 1., 1.],
       [1., 1., 1.]])

In [8]:
n3 = np.zeros([2,3])
n3

array([[0., 0., 0.],
       [0., 0., 0.]])

In [9]:
t2 = torch.ones(2,3)
t2

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [20]:
print(t2.shape, t2.dtype, t2.type(), t2.device)

torch.Size([2, 3]) torch.float32 torch.FloatTensor cpu


In [11]:
t3 = torch.zeros(2,3)
t3

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [12]:
# numpy로 만든 배열을 텐서로 변환
t4 = torch.from_numpy(n3)
t4

tensor([[0., 0., 0.],
        [0., 0., 0.]], dtype=torch.float64)

- 넘파이와 파이토치로 텐서간의 형변환이 아주쉽다

In [14]:
t5 = torch.ones(3, 3, 3)
t5

tensor([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]])

In [22]:
print(t5.shape, t5.dtype, t5.type(), t5.device)
print(t5.size())

torch.Size([3, 3, 3]) torch.float32 torch.FloatTensor cpu
torch.Size([3, 3, 3])


In [ ]:
# 랜던값, randn()은 정규값
t6 = torch.randn(2,3)
t6

tensor([[-1.7216, -0.7053, -0.3066],
        [-0.0521, -1.0231, -0.2067]])

In [23]:
# cpu에 있는 텐서를 cuda로 이동
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [24]:
# t5(cpu)에 있는 텐서를 cuda로 변수명을 다르게하면 복사, 변수명을 똑같이하면 이동
t5 = t5.to(device=device)

In [25]:
print(t5.shape, t5.dtype, t5.type(), t5.device)

torch.Size([3, 3, 3]) torch.float32 torch.cuda.FloatTensor cuda:0


- cuda(GPU사용하는 것): 0(그래픽카드가 다수일수 있음)
- 파이토치의 텐서사용 == 넘파이 배열사용

#### GPU사용법

In [ ]:
# cpu사용
import torch.nn

# 신경망 모델 생성 - 3개의 입력을 넣어서 하나의 출력을 내는 신경망
model = torch.nn.Linear(3, 1)

sample_input = torch.tensor([[1.0, 2.0, 3.0]])
output= model(sample_input)

output

tensor([[0.9887]], grad_fn=<AddmmBackward0>)

In [ ]:
# CUDA사용 GPU사용해서 데이터가 많은상테에서 자료가 많으면 학습을 빠르고 정확하게 나온다
import torch.nn

# 신경망 모델 생성 - 3개의 입력을 넣어서 하나의 출력을 내는 신경망
model = torch.nn.Linear(3, 1)
model.to(device) # 모델 객체도  GPU로 전달

sample_input = torch.tensor([[1.0, 2.0, 3.0]]).to(device)
output= model(sample_input)

output

tensor([[2.1499]], device='cuda:0', grad_fn=<AddmmBackward0>)

### Pytorch 모델 학습
- torch.nn.Model 로 신경망 만들기  - 텐서플로(함수)우와 차이점 확인
- 손실 함수, 옵티마이저 설정 학습
- 학습 루프 기본 구조 학습

#### 기본 구조 추가

In [ ]:
# 파이토치 모듈 
import torch
import torch.nn as nn
import torch.nn.functional as F  #보통 F로 사용

# 모델선언()
class SimpleNet(nn.Module):     # nn.Model 클래스를 상속해서 클래스 생성
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Linear는 Dense Layer(케라스의 Dense()와 동일)
        self.fc1 = nn.Linear(4, 16) # input : 4개 입력, output: 16개 뉴런(차원) 출력갯수와 밑의 입력갯수는 같아야 출력이 가능하다
        self.fc2 = nn.Linear(16, 3) # input : 16개 입력, output 3개 클래스(3가지 분류 예측) 

    def forward(self, x):   # forward는 실제 연산이 수행되는 함수(자동호출)
        x = F.relu(self.fc1(x)) # 첫번쨰 레이어 통과 - 활성화함수(시그모이드, 소프트맥스 렐루 모두 존재 )
        x = self.fc2(x)         # 두번쨰 레이어 통과 - 출력층(필요시 softmax, loss함수 )
        return x

- 텐서플로우(케라스)는 중간층에서 입력값을 설정하지 않아도 됨
- 파이토치는 입력값, 출력값을 모두 설정해야함

In [34]:
# 모델확인
model = SimpleNet()
model

SimpleNet(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=3, bias=True)
)

In [35]:
# 샘플입력값
x = torch.rand(1, 4)
x

tensor([[0.8032, 0.1549, 0.3077, 0.7621]])

In [36]:
out = model(x)
out

tensor([[-0.2005, -0.3674,  0.0448]], grad_fn=<AddmmBackward0>)

#### 손실함수/옵티마이저 설정

In [37]:
# 분류(마지막 출력 3개)CrossEntropyLoss - 
criterion = nn.CrossEntropyLoss()

# 옵티마이저(Adma 권장)
# parameters는 위의 모델 확인에 4, 16 True를 말한다 
oprimizer = torch.optim.Adam(model.parameters(), lr=0.01) # lr(learing rate) 학습률 : 보통0.01~0.0001 사이포 지정

#### 1회 Epoch 학습

In [ ]:
y = torch.tensor([1])

#Forward
# 예측
outputs = model(x)
# 손실 계산
loss = criterion(outputs,y)

#Backward(역전파) : 뉴런을 통과한 값이 다시 다음 훈련의 입력으로 사용
# 기울기 초기화
oprimizer.zero_grad() # 이전단계 계산된 gradient(기울기/가중치) 초기화
#역전파 계산
loss.backward() # 손실을 기준으로 각 파라미터에대한 gradient 계산(오차 역전파)
# 파라미터 업데이트
oprimizer.step() # 계산한 gradient를 이용, 파라미터 업데이터(학습훈련)

# 현재 손실값을 출력
loss.item()

1.2423126697540283

### 데이터셋, 데이터로더
- `Dataset`, `DataLoader` 학습
- 내장 데이터셋
- 커스텀 데이터셋 만들기
- 배치학습 처리 

#### 데이터겟, 데이터로더란?
- Dataset - 데이터를 불러오는 방법
- DataLofder - 데이터를 배치 단위로 나누기, 셔플, 병렬처리 방법

#### 붓꽃(Iris) 데이터셋 사용

#### 추가
- 머신러닝, 딥러닝에 사용되는 데이터샘플 종류예제

In [43]:
from sklearn.datasets import load_breast_cancer, load_wine # 11개 데이터셋
import seaborn as sns

datas = sns.load_dataset('titanic') # https://github.com/mwaskom/seaborn-data 확인

from torchvision.datasets import FashionMNIST, MNIST, CIFAR10 # 이미지 데이터셋
from keras.datasets import mnist, fashion_mnist, boston_housing, cifar10,cifar100,imdb  # 이미지, 텍스트 데이터셋

In [44]:
datas

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [45]:
# 데이터셋 처리 모듈 로드
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch

In [48]:
# 데이터 로드
iris = load_iris()

## 입력(특성)데이터 X, 타겟 y == input, target
X = iris['data']
y = iris['target']

print(X.shape, y.shape)  # numpy 데이터

(150, 4) (150,)


In [50]:
# sepal length(cm), sepal width(cm), petal(꽃잎) length(cm), petal width(cm)

X[0,:]

array([5.1, 3.5, 1.4, 0.2])

In [51]:
# 0(setosa), 1(versicolor), 2(virginical)
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [52]:
# 표준화(정규화)
scaler = StandardScaler() # 사이킷런의 표준정규화 클래스 사용
X = scaler.fit_transform(X)

In [54]:
X[0, :]

array([-0.90068117,  1.01900435, -1.34022653, -1.3154443 ])

In [56]:
# 훈련/검증세트 분리
train_scaled, test_scaled, train_target, test_target = train_test_split(X, y ,test_size=0.2, random_state=42)

In [57]:
# 훈련/ 검증세트 분리
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target,test_size=0.2, random_state=42)

In [58]:
print(train_scaled.shape, val_scaled.shape, test_scaled.shape)

(96, 4) (24, 4) (30, 4)


In [59]:
train_target

array([2, 1, 1, 0, 2, 2, 2, 0, 1, 2, 2, 1, 2, 1, 1, 0, 0, 1, 2, 0, 0, 2,
       2, 1, 2, 1, 1, 0, 2, 0, 1, 2, 0, 2, 0, 0, 2, 0, 2, 1, 2, 0, 0, 0,
       0, 2, 0, 0, 1, 2, 1, 0, 1, 2, 1, 2, 2, 2, 2, 1, 1, 0, 0, 0, 2, 0,
       0, 0, 2, 1, 2, 1, 2, 1, 1, 0, 0, 2, 1, 0, 2, 1, 2, 1, 2, 1, 1, 2,
       1, 1, 0, 2, 0, 1, 0, 0])

#### 커스텀 데이터셋 만들기

In [ ]:
class IrisDataset(Dataset):
    def __init__(self,X,y):
        self.X = torch.tensor(X, dtype=torch.float32) # 특정(입력)값은 실수
        self.y = torch.tensor(y, dtype=torch.float)    # CorssEntropyLoss에서는 long타입 필요

    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        return self.X[index], self.y[index]
                   

In [ ]:
# 커스텀 데이터셋으로 생성 - 입력(특성)과 타겟을 하나로 묶음 DataLoader 묶기
train_dataset = IrisDataset(train_scaled, train_target)
val_dataset = IrisDataset(val_scaled, val_target)

In [ ]:
# 데이터로 생성 - 배치학습, 셔플 ,병렬데이터 로딩
train_loader = DataLoader(train_dataset,batch_size=16, shuffle='True')
val_loader = DataLoader(val_dataset, batch_size=16)

- 실제 모델훈련에서는 Dataloader를 사용